# Download Planet Basemap Scenes for ARTS Using Orders API
This script downloads 2024 Planet Basemap grids to cover all ARTS v.6.0.0 polygons.

In [2]:
import os
import json
import requests
import urllib.request
import numpy as np
import pandas as pd
import geopandas as gpd
import shapely as shp
from pprint import pprint
import math
import time
import ast
import re
import sys
from datetime import datetime

In [9]:
# Get Planet API Key
%load_ext dotenv
%dotenv

api_key = os.getenv('PL_BM_API_KEY')
globus_key = os.getenv("GLOBUS_PL_ORDERS_KEY")
globus_access_key_id = os.getenv("GLOBUS_ACCESS_KEY_ID")

The dotenv extension is already loaded. To reload it, use:
  %reload_ext dotenv


In [4]:
# setup session
session = requests.Session()

# authenticate
session.auth = (api_key, "")


# Import Data

In [6]:
grids_filtered = gpd.read_file('../data/planet_grids_2024_artsv.6.0.0.geojson')
grids_filtered

,year,id,grid_column,grid_row,basemap_name,delivery_location,link,geometry
0,2024,0-1515,0,1515,global_quarterly_2024q3_mosaic,planet_basemaps/global_quarterly/2024/q3/0/1515/,https://link.planet.com/basemaps/v1/mosaics/44...,"POLYGON ((-1950841.755 1938908.115, -1944872.9..."
1,2024,0-1516,0,1516,global_quarterly_2024q3_mosaic,planet_basemaps/global_quarterly/2024/q3/0/1516/,https://link.planet.com/basemaps/v1/mosaics/44...,"POLYGON ((-1944872.974 1932975.846, -1938922.4..."
2,2024,0-1549,0,1549,global_quarterly_2024q3_mosaic,planet_basemaps/global_quarterly/2024/q3/0/1549/,https://link.planet.com/basemaps/v1/mosaics/44...,"POLYGON ((-1757802.562 1747049.776, -1752423.2..."
3,2024,0-1553,0,1553,global_quarterly_2024q3_mosaic,planet_basemaps/global_quarterly/2024/q3/0/1553/,https://link.planet.com/basemaps/v1/mosaics/44...,"POLYGON ((-1736383.836 1725762.072, -1731069.9..."
4,2024,1-1532,1,1532,global_quarterly_2024q3_mosaic,planet_basemaps/global_quarterly/2024/q3/1/1532/,https://link.planet.com/basemaps/v1/mosaics/44...,"POLYGON ((-1857451.331 1834795.702, -1851767.6..."
...,...,...,...,...,...,...,...,...
23685,2024,2047-1516,2047,1516,global_quarterly_2024q3_mosaic,planet_basemaps/global_quarterly/2024/q3/2047/...,https://link.planet.com/basemaps/v1/mosaics/44...,"POLYGON ((-1938933.535 1938933.535, -1933001.1..."
23686,2024,2047-1533,2047,1533,global_quarterly_2024q3_mosaic,planet_basemaps/global_quarterly/2024/q3/2047/...,https://link.planet.com/basemaps/v1/mosaics/44...,"POLYGON ((-1840509.164 1840509.164, -1834877.3..."
23687,2024,2047-1536,2047,1536,global_quarterly_2024q3_mosaic,planet_basemaps/global_quarterly/2024/q3/2047/...,https://link.planet.com/basemaps/v1/mosaics/44...,"POLYGON ((-1823665.148 1823665.148, -1818084.7..."
23688,2024,2047-1546,2047,1546,global_quarterly_2024q3_mosaic,planet_basemaps/global_quarterly/2024/q3/2047/...,https://link.planet.com/basemaps/v1/mosaics/44...,"POLYGON ((-1768621.76 1768621.76, -1763209.464..."


In [7]:
imagery_year = '2024'

In [ ]:
start = datetime.now()
print("Downloads started at ", start)
delivery_directory = (
    "/global_quarterly/test_delivery"
)

for index, row in grids_filtered[0:1].iterrows():
    basemap_name = row["basemap_name"]
    item_id = [row["id"]]  # only deliver a single item at a time
    
    print("-------------------------------------")
    print(index, ": ", item_id[0])

    order_info = {
        "name": order_name,
        "source_type": "basemaps",
        "products": [{"mosaic_name": basemap_name, "quad_ids": item_id}],
        "delivery": {
            "s3_compatible": {
                "endpoint": "https://rci-s3.msu.montana.edu",
                "bucket": "planet",
                "region": "us-east-1",
                "access_key_id": globus_access_key_id,
                "secret_access_key": globus_key,
                "use_path_style": False
            }
        }
    }
    
    # Place order
    print(
        "Placing order for delivery to ", dir_path
    )
    request = requests.post('https://api.planet.com/compute/ops/orders/v2',
                            auth=(api_key, ''),
                            json=order_info)
    print(str(request))
    if str(request) in ['<Response [400]>', '<Response [401]>']:
        sys.exit("Invalid request or credentials. Check your request.")
    if str(request) == '<Response [409]>':
        while str(request) == '<Response [409]>':
            print("Concurrency error. Retrying in 30 seconds...")
            time.sleep(30)
            request = requests.post('https://api.planet.com/compute/ops/orders/v2',
                                    auth=(api_key, ''),
                                    json=order_info)
    if str(request) == '<Response [500]>': # server error, try again
        while str(request) == "<Response [500]>":
            print("Server error. Retrying in 30 seconds...")
            time.sleep(30)
            request = requests.post(
                "https://api.planet.com/compute/ops/orders/v2",
                auth=(api_key, ""),
                json=order_info,
            )
    if str(request) == '<Response [202]>':
        print('Order has been placed.')

        
end = datetime.now()
print("Downloads finished at ", end)
total_time = end - start
print("Downloading took ", total_time)


Downloads started at  2026-02-25 09:32:15.124553
-------------------------------------
0 :  0-1515
Placing order for delivery to  planet_basemaps/global_quarterly/2024/q3/0/
<Response [202]>
Order has been placed.
Downloads finished at  2026-02-25 09:32:17.694046
Dowloading  23690  grids took  0:00:02.569493
